# Módulo F3 · Estadística para la toma de decisiones

**Caso:** una app de reparto quiere estimar el **importe medio del pedido** y decidir si un cambio de interfaz lo ha aumentado. Solo dispone de una muestra de pedidos. ¿Cuánto podemos fiarnos de lo que vemos?

Objetivos:
1. Reconocer las distribuciones más habituales.
2. Entender la ley de los grandes números y el teorema central del límite (TCL) mediante simulación.
3. Construir e interpretar intervalos de confianza (fórmula y *bootstrap*).
4. Detectar la paradoja de Simpson y distinguir correlación de causalidad.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
rng = np.random.default_rng(7)

## 1. Distribuciones que verás constantemente

| Distribución | Modela… | Ejemplo |
|---|---|---|
| Bernoulli/Binomial | éxitos en *n* intentos | conversiones de una campaña |
| Poisson | conteos por intervalo | pedidos por hora |
| Normal | sumas de muchos efectos pequeños | errores de medida |
| Exponencial | tiempos de espera | minutos hasta el siguiente pedido |
| Log-normal | cantidades positivas asimétricas | importe de un pedido, salarios |

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.2))
axes[0].hist(rng.binomial(20, 0.1, 10_000), bins=range(0, 10), rwidth=0.9); axes[0].set_title("Binomial(20, 0.1)")
axes[1].hist(rng.poisson(4, 10_000), bins=range(0, 15), rwidth=0.9); axes[1].set_title("Poisson(4)")
axes[2].hist(rng.exponential(5, 10_000), bins=50); axes[2].set_title("Exponencial(media 5)")
axes[3].hist(rng.lognormal(3.2, 0.5, 10_000), bins=50); axes[3].set_title("Log-normal")
plt.tight_layout(); plt.show()

## 2. La "población" de pedidos

Simulamos 1 millón de pedidos (la población, que en la vida real **nunca** observamos completa). Su media real es la cantidad que queremos estimar.

In [ ]:
poblacion = rng.lognormal(mean=3.2, sigma=0.5, size=1_000_000)
mu_real = poblacion.mean()
print(f"Media poblacional (desconocida en la práctica): {mu_real:.2f} €")

## 3. Ley de los grandes números
La media muestral se acerca a la media real a medida que crece la muestra.

In [ ]:
muestra_grande = rng.choice(poblacion, 5_000)
medias_acumuladas = np.cumsum(muestra_grande) / np.arange(1, len(muestra_grande) + 1)
plt.plot(medias_acumuladas)
plt.axhline(mu_real, color="red", ls="--", label="media real")
plt.xscale("log"); plt.xlabel("tamaño de muestra"); plt.ylabel("media muestral"); plt.legend()
plt.title("Ley de los grandes números"); plt.show()

## 4. Teorema central del límite
Aunque los pedidos son asimétricos, la **distribución de la media muestral** se parece cada vez más a una normal, con desviación típica $\sigma/\sqrt{n}$ (el *error estándar*).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
for ax, n in zip(axes, [2, 10, 100]):
    medias = rng.choice(poblacion, (5_000, n)).mean(axis=1)
    ax.hist(medias, bins=60, density=True, alpha=0.7)
    x = np.linspace(medias.min(), medias.max(), 200)
    ax.plot(x, stats.norm.pdf(x, mu_real, poblacion.std() / np.sqrt(n)), "r")
    ax.set_title(f"n = {n}  |  EE teórico = {poblacion.std() / np.sqrt(n):.2f}")
plt.tight_layout(); plt.show()

## 5. Intervalos de confianza

Con una sola muestra de 200 pedidos, el intervalo al 95 % es
$$\bar x \pm t_{0.975,\,n-1}\cdot \frac{s}{\sqrt n}$$

In [ ]:
muestra = rng.choice(poblacion, 200)
media, ee = muestra.mean(), stats.sem(muestra)
ic = stats.t.interval(0.95, df=len(muestra) - 1, loc=media, scale=ee)
print(f"Media muestral: {media:.2f} € | IC 95 %: [{ic[0]:.2f}, {ic[1]:.2f}]")

**¿Qué significa "95 %"?** Si repitiéramos el muestreo muchas veces, el 95 % de los intervalos construidos así contendrían la media real. Comprobémoslo:

In [ ]:
aciertos, intervalos = 0, []
for _ in range(100):
    m = rng.choice(poblacion, 200)
    lo, hi = stats.t.interval(0.95, df=199, loc=m.mean(), scale=stats.sem(m))
    intervalos.append((lo, hi))
    aciertos += lo <= mu_real <= hi

plt.figure(figsize=(12, 3))
for i, (lo, hi) in enumerate(intervalos):
    plt.plot([i, i], [lo, hi], color="tab:blue" if lo <= mu_real <= hi else "red")
plt.axhline(mu_real, color="black", ls="--")
plt.title(f"{aciertos} de 100 intervalos contienen la media real"); plt.show()

### Bootstrap: intervalos sin fórmulas
Remuestreamos **con reemplazo** nuestra propia muestra miles de veces. Sirve para cualquier estadístico (mediana, percentil 90, ratio…), no solo la media.

In [ ]:
def bootstrap(datos, estadistico, n_boot=5_000, rng=rng):
    idx = rng.integers(0, len(datos), (n_boot, len(datos)))
    return np.array([estadistico(datos[i]) for i in idx])

boot_mediana = bootstrap(muestra, np.median)
lo, hi = np.percentile(boot_mediana, [2.5, 97.5])
print(f"Mediana: {np.median(muestra):.2f} € | IC 95 % bootstrap: [{lo:.2f}, {hi:.2f}]")
print(f"Mediana poblacional: {np.median(poblacion):.2f} €")

## 6. Correlación no es causalidad: la paradoja de Simpson

Dos versiones de la app (A y B). Mira la tasa de conversión global y luego por tipo de dispositivo.

In [ ]:
datos = pd.DataFrame({
    "version":     ["A", "A", "B", "B"],
    "dispositivo": ["móvil", "escritorio", "móvil", "escritorio"],
    "visitas":     [8000, 2000, 2000, 8000],
    "compras":     [400, 300, 80, 1040],
})
datos["tasa"] = datos["compras"] / datos["visitas"]
print(datos, "\n")
glob = datos.groupby("version")[["visitas", "compras"]].sum()
print("Tasa global:\n", (glob["compras"] / glob["visitas"]).round(3), sep="")

B gana en el global (11,2 % frente a 7 %)… pero **A gana en cada segmento**. La causa: B se mostró sobre todo en escritorio, donde todos convierten más. El dispositivo es una **variable de confusión**. Moraleja: aleatoriza la asignación (prueba A/B, módulo 04) o controla las variables relevantes.

## Ejercicios

1. Repite el experimento de cobertura con muestras de `n = 15`. ¿Sigue siendo ~95 %? ¿Por qué podría fallar con datos tan asimétricos?
2. Calcula por bootstrap un IC 95 % para el **percentil 90** del importe. ¿Por qué sería útil para planificar repartidores?
3. ¿Qué tamaño de muestra necesitas para que el IC de la media tenga una semianchura de ±0,50 €? (Pista: despeja $n$ de $1{,}96\,s/\sqrt n = 0{,}5$.)
4. Inventa otro ejemplo de paradoja de Simpson en tu sector y escríbelo en una tabla como la anterior.

In [ ]:
# TU CÓDIGO AQUÍ